# 02 保存snapshotからPIT信号を作る

## Goal

共通の ResearchStore と build_pit_signal_table を使い、各判断時刻に実際に観測済みの合成snapshotだけからモメンタムとボラティリティを計算します。

## Setup

- 入力: 01と同じ固定合成デモのDEMO:ALPHA、lockbox開始前46足
- snapshot: 各足の判断時刻に観測した1本ずつを一時ディレクトリへ保存し、終了後に削除
- 通貨/調整方式: USD / raw。時刻: UTC。足終端と判断時刻は明示
- 未来1期リターンは評価用ラベルです。予測入力に混ぜません
- 実市場の履歴を後から保存しただけではPITと見なしません。このノートは合成の時点別snapshotを用いた契約例です

In [ ]:
from pathlib import Path
import pandas as pd
from tempfile import TemporaryDirectory

import plotly.graph_objects as go
from market_research.research.signals import build_pit_signal_table
from market_research.services import build_demo_run
from market_research.storage import CacheKey, ResearchStore

run = build_demo_run()
lockbox_start = run.prices.index[46].to_pydatetime()
selected_bars = tuple(
    bar for bar in run.bars
    if bar.instrument.instrument_id == "DEMO:ALPHA"
    and bar.available_at < lockbox_start
)
print(f"run_id={run.run_id}, lockbox_start={lockbox_start.isoformat()}, selected={len(selected_bars)}")

## Steps

### 1. 合成足を観測時刻付きで保存する

各snapshotにはその判断時点の足を1本だけ含めます。保存と読取はローカルのみです。snapshot IDはコードから生成せず、保存APIが返す値を使います。

In [ ]:
with TemporaryDirectory(prefix="market-research-notebook-") as directory:
    with ResearchStore(Path(directory)) as store:
        snapshot_ids = []
        for bar in selected_bars:
            key = CacheKey(
                bar.provider, "prices", bar.instrument.instrument_id,
                bar.interval, bar.instrument.currency, bar.adjustment,
            )
            raw = f"synthetic:{bar.bar_end.isoformat()}:{bar.close:.10f}".encode()
            saved = store.save(
                key, raw, observed_at=bar.observed_at, prices=(bar,),
            )
            snapshot_ids.append(saved.snapshot_id)
        signals = build_pit_signal_table(
            store,
            instrument_id="DEMO:ALPHA",
            snapshot_ids=tuple(snapshot_ids),
            decision_times=tuple(bar.available_at for bar in selected_bars),
            expected_bar_ends=tuple(bar.bar_end for bar in selected_bars),
            currency="USD",
            adjustment="raw",
            momentum_window=3,
            volatility_window=3,
            horizon=1,
        )
print(f"mode={signals.mode}, snapshot_count={len(signals.snapshot_ids)}")
signals.table.tail(8)

### 2. 信号の推移を見る

両系列は比率です。先頭の不足期間は欠損のままにし、前値で補完しません。線は未来ラベルを含みません。

In [ ]:
figure = go.Figure()
for name, dash in (("momentum", "solid"), ("volatility", "dash")):
    figure.add_scatter(
        x=signals.table.index,
        y=signals.table[name],
        mode="lines",
        name=name,
        line={"dash": dash},
    )
figure.update_layout(
    title="合成DEMO:ALPHAの時点別信号 (3期窓、lockbox前)",
    xaxis_title="判断時刻 (UTC)",
    yaxis_title="比率",
    template="plotly_white",
)
figure

## Checks

ラベルの使用可能時刻は次の判断時刻です。末尾の未来リターンは存在しないため欠損です。lockbox以降のsnapshot IDを入力に渡していません。

In [ ]:
assert signals.mode == "point_in_time"
assert len(signals.snapshot_ids) == len(selected_bars) == 46
assert signals.table.index.max() < lockbox_start
assert signals.table["label_available_at"].dropna().lt(lockbox_start).all()
assert signals.table.iloc[0]["label_available_at"] == signals.table.index[1]
assert pd.isna(signals.table.iloc[-1]["label"])
assert pd.isna(signals.table.iloc[-1]["label_available_at"])
print("未来ラベルの時刻・末尾欠損・lockbox境界を確認")

## Next Steps

03では同じsnapshot生成手順を単独で実行し、訓練ラベルと検証期間を分離した比較を行います。ここに表示したラベルを信号として使わないでください。